<a href="https://colab.research.google.com/github/sharifa-15/Fraud-Detection-Financial-Transactions/blob/main/Fraud_Detection_in_FInancial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense


In [ ]:
# Load dataset
df = pd.read_csv("fraudTest.csv")

# Check imbalance
print(df['gender'].value_counts())
print(df['merchant'].value_counts())

gender
F    127413
M    105349
Name: count, dtype: int64
merchant
fraud_Kilback LLC                        782
fraud_Schumm PLC                         679
fraud_Dickinson Ltd                      654
fraud_Cormier LLC                        634
fraud_Boyer PLC                          632
                                        ... 
fraud_Kessler Group                      132
fraud_Ankunding-Carroll                  131
fraud_Schroeder, Wolff and Hermiston     129
fraud_Larson, Quitzon and Spencer        127
fraud_Ritchie, Bradtke and Stiedemann    126
Name: count, Length: 693, dtype: int64


In [ ]:
print(df.head())
print(df['is_fraud'].value_counts())

   Unnamed: 0 trans_date_trans_time            cc_num  \
0           0   2020-06-21 12:14:25  2291163933867244   
1           1   2020-06-21 12:14:33  3573030041201292   
2           2   2020-06-21 12:14:53  3598215285024754   
3           3   2020-06-21 12:15:15  3591919803438423   
4           4   2020-06-21 12:15:17  3526826139003047   

                               merchant        category    amt   first  \
0                 fraud_Kirlin and Sons   personal_care   2.86    Jeff   
1                  fraud_Sporer-Keebler   personal_care  29.84  Joanne   
2  fraud_Swaniawski, Nitzsche and Welch  health_fitness  41.28  Ashley   
3                     fraud_Haley Group        misc_pos  60.05   Brian   
4                 fraud_Johnston-Casper          travel   3.19  Nathan   

       last gender                       street  ...      lat      long  \
0   Elliott      M            351 Darlene Green  ...  33.9659  -80.9355   
1  Williams      F             3638 Marsh Union  ...  40.3207 

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical variables
categorical_cols = ['merchant','category','gender','job','city','state']
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col].astype(str))

# Drop rows where 'is_fraud' is NaN
df.dropna(subset=['is_fraud'], inplace=True)

# Features & labels
X = df.drop(['is_fraud','trans_date_trans_time','first','last','street','dob','trans_num'], axis=1)
y = df['is_fraud']

# Scale numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_res.value_counts())


Before SMOTE: is_fraud
0.0    185398
1.0       810
Name: count, dtype: int64
After SMOTE: is_fraud
0.0    185398
1.0    185398
Name: count, dtype: int64


In [ ]:
iso = IsolationForest(contamination=0.01, random_state=42)
y_pred_iso = iso.fit_predict(X_test)

# Convert -1 to fraud (1), 1 to normal (0)
y_pred_iso = [1 if x==-1 else 0 for x in y_pred_iso]

print("Isolation Forest Results:")
print(classification_report(y_test, y_pred_iso))
print(confusion_matrix(y_test, y_pred_iso))


Isolation Forest Results:
              precision    recall  f1-score   support

         0.0       1.00      0.99      0.99     46351
         1.0       0.01      0.02      0.01       202

    accuracy                           0.99     46553
   macro avg       0.50      0.50      0.50     46553
weighted avg       0.99      0.99      0.99     46553

[[45889   462]
 [  198     4]]


In [ ]:
input_dim = X_train_res.shape[1]
input_layer = Input(shape=(input_dim,))
encoder = Dense(32, activation="relu")(input_layer)
encoder = Dense(16, activation="relu")(encoder)
decoder = Dense(32, activation="relu")(encoder)
decoder = Dense(input_dim, activation="linear")(decoder)

autoencoder = Model(inputs=input_layer, outputs=decoder)
autoencoder.compile(optimizer='adam', loss='mse')

# Train only on normal transactions
X_train_norm = X_train_res[y_train_res==0]
autoencoder.fit(X_train_norm, X_train_norm,
                epochs=20, batch_size=64, shuffle=True, validation_split=0.2)

# Reconstruction error
reconstructions = autoencoder.predict(X_test)
mse = np.mean(np.power(X_test - reconstructions, 2), axis=1)

# Threshold
threshold = np.percentile(mse, 95)
y_pred_ae = [1 if e > threshold else 0 for e in mse]

print("AutoEncoder Results:")
print(classification_report(y_test, y_pred_ae))
print(confusion_matrix(y_test, y_pred_ae))


Epoch 1/20
2318/2318 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 0.0859 - val_loss: 0.0039
Epoch 2/20
2318/2318 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0013 - val_loss: 5.4344e-04
Epoch 3/20
2318/2318 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 5.8114e-04 - val_loss: 5.0097e-04
Epoch 4/20
2318/2318 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 5.9423e-04 - val_loss: 4.6239e-04
Epoch 5/20
2318/2318 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 7.8115e-04 - val_loss: 4.8876e-04
Epoch 6/20
2318/2318 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 6.2778e-04 - val_loss: 3.2174e-04
Epoch 7/20
2318/2318 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 2.7555e-04 - val_loss: 1.8694e-04
Epoch 8/20
2318/2318 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 2.6114e-04 - val_loss: 1.0233e-04
Epoch 9/20
2318/2318 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 3.3706e-04 - val_loss: 9.8970e-05
Epoch 10/20
2318/2318 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 1.0456e-04 - val_loss: 1.7064e-04
Epoch 11/20
2318/2318 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/s

In [ ]:
results = pd.DataFrame({
    'TransactionID': df.index[:len(y_test)],
    'Fraud_Pred_IF': y_pred_iso,
    'Fraud_Pred_AE': y_pred_ae,
    'Actual': y_test.values
})

results.to_csv("fraud_predictions.csv", index=False)
